In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/pranaysingh25/mnist-data/mnist_test.csv
/kaggle/input/datasets/pranaysingh25/mnist-data/mnist_train.csv


**NOTE:** This notebook was originally created on Kaggle for using GPU optimizations from Kaggle (GPU T4 x2) for faster image processing. If you want to work with it locally, change the Kaggle file paths of the train and test datasets to the downloaded data folder's mnist files train and test paths instead: (../data/mnist_train.csv) and (../data/mnist_test.csv).
---

The GPU’s ability to use massive parallel processing is precisely why it dominates when handling images or 2D grids like spectrograms (image where the X-axis is Time and the Y-axis is Frequency).
A CPU is built like a high-performance sports car. It has a few extremely fast, powerful cores (typically 4 to 16). It is designed for serial processing—meaning it takes a task, finishes it incredibly quickly, and moves to the next one.
A GPU is built like a fleet of thousands of delivery trucks. Its individual cores are much simpler and slower than a CPU core, but it has thousands of them working simultaneously (parallel processing).

Images are inherently independent at the pixel level. If you want to multiply every pixel in an image by a filter weight, the calculation for the top-left pixel doesn't care about what the bottom-right pixel is doing.

Because of this independence, the GPU can say:

Core 1: Calculate Pixel 1

Core 2: Calculate Pixel 2

Core 3: Calculate Pixel 3

...

Core 2000: Calculate Pixel 2000

Instead of waiting in line, thousands of pixel calculations happen at the exact same millisecond.

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

2026-05-04 03:13:52.752308: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777864432.971315      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777864433.035493      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777864433.552122      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777864433.552156      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777864433.552159      23 computation_placer.cc:177] computation placer alr

In [3]:
# 1. Preprocessing Utility Function
def load_and_preprocess_mnist(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # Separate labels and features
    # (Assuming first column is 'label')
    Y_train = train_df.iloc[:, 0].values
    X_train = train_df.iloc[:, 1:].values / 255.0
    
    Y_test = test_df.iloc[:, 0].values
    X_test = test_df.iloc[:, 1:].values / 255.0
    
    # Reshape for CNN: (Samples, Height, Width, Channels)
    X_train = X_train.reshape(-1, 28, 28, 1)
    X_test = X_test.reshape(-1, 28, 28, 1)
    
    return X_train, Y_train, X_test, Y_test

In [4]:
# 2. Set Paths based on your Kaggle Input sidebar
train_path = '/kaggle/input/datasets/pranaysingh25/mnist-data/mnist_train.csv'
test_path = '/kaggle/input/datasets/pranaysingh25/mnist-data/mnist_test.csv'

# 3. Execute Preprocessing
X_train, Y_train, X_test, Y_test = load_and_preprocess_mnist(train_path, test_path)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (60000, 28, 28, 1)
X_test shape: (10000, 28, 28, 1)


In [5]:
# Define the CNN Architecture
model = models.Sequential([
    # First Convolutional Block: Extracts basic edges/shapes
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    
    # Second Convolutional Block: Extracts complex patterns
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Transition to Classification
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2), # Prevents overfitting (the 27% accuracy issue)
    layers.Dense(10, activation='softmax')
])

# Compile the Model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Show the summary of our creation
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1777864462.717368      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777864462.722951      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,930 (476.29 KB)

 Trainable params: 121,930 (476.29 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Train the CNN
history = model.fit(
    X_train, Y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, Y_test)
)

Epoch 1/10


I0000 00:00:1777864465.820132      70 service.cc:152] XLA service 0x7da08000a450 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777864465.820190      70 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777864465.820196      70 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777864466.146412      70 cuda_dnn.cc:529] Loaded cuDNN version 91002


 54/938 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.4203 - loss: 1.6965

I0000 00:00:1777864468.918113      70 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


938/938 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.8568 - loss: 0.4520 - val_accuracy: 0.9846 - val_loss: 0.0466
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9797 - loss: 0.0669 - val_accuracy: 0.9864 - val_loss: 0.0374
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9848 - loss: 0.0477 - val_accuracy: 0.9901 - val_loss: 0.0285
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9886 - loss: 0.0366 - val_accuracy: 0.9905 - val_loss: 0.0279
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9909 - loss: 0.0297 - val_accuracy: 0.9907 - val_loss: 0.0279
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9920 - loss: 0.0247 - val_accuracy: 0.9915 - val_loss: 0.0287
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9928 - loss: 0.0214 - val_accuracy: 0.9907 - val_loss: 0.0307
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9946 - loss: 0.0164 - val_accuracy: 0.9916 - val

For a CNN, we must keep the 2D shape (28 x 28) so the filters can detect spatial patterns like edges and curves.

**That 99.28% validation accuracy is incredible. It proves that the Convolutional layers are doing exactly what they were designed to do: identifying the spatial geometry of the numbers, rather than just looking at raw flattened pixel intensities. You've officially surpassed the performance of your scratch-built model by a wide margin.**